# Reflection Agents: Basic Reflection, Reflexion, LATS

From https://www.langchain.com/blog/reflection-agents — three ways to have
an agent critique and improve its own output instead of returning the first
draft:

1. **Basic Reflection** — generate/reflect loop. A generator drafts a
   response, a reflector critiques it as a teacher would, the generator
   redrafts. Purely self-critique, no external grounding.
2. **Reflexion** — draft -> execute_tools -> revise. Grounds the critique in
   real tool results (citations, missing/superfluous content) instead of the
   model's own opinion, but follows one fixed trajectory.
3. **LATS** (Language Agent Tree Search) — Monte Carlo tree search over
   candidate trajectories (select via UCT, expand, reflect/score, backpropagate),
   exploring several paths instead of one linear retry chain.

None of these exist in psychscanner today, so `psychscanner.agents.reflection_agents`
builds all three directly on LangGraph and adapts them to the `ScanningAgent`
contract. Basic Reflection and LATS need only plain chat completions
(`llama3.2:3b`); Reflexion additionally needs tool calling.

In [1]:
from pathlib import Path

from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

import psychscanner as psy
from psychscanner.agents import make_basic_reflection_agent, make_lats_agent, make_reflexion_agent
from psychscanner.memories import llm_chat_model
from psychscanner.task_runner import TaskRunner

model = llm_chat_model(model="llama3.2:3b", family="ollama", parameters={"temperature": 0})
print("PsychScanner successfully imported!")

--<api key>-- warning: OLLAMA_API_KEY not set; proceeding without explicit api_key for family 'ollama'


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model='llama3.2:3b' temperature=0.0


PsychScanner successfully imported!


In [2]:
def run_one(agent, stimulus_text, system_message="You are a helpful assistant."):
    tasktrials = {"trials": [{"trcode": "t1", "stimulus": HumanMessage(content=stimulus_text),
                              "tasktype": "x", "parser": None, "fb": False}]}
    runner = TaskRunner(
        scanning_agent=agent, trace_cfg={"trial": "tut-", "task": "tut-task"},
        system_message=system_message, tasktrials=tasktrials, chain_type="item", hmsg="stimulus",
    )
    return runner.execute()[0]["pred_resp"].content

## 1. Basic Reflection

In [ ]:
reflection_agent = make_basic_reflection_agent(model, max_messages=4)  # 2 generate + 1 reflect turn
print(run_one(reflection_agent, "Write a one-sentence summary of what an n-back task measures."))

## 2. Reflexion

Grounds the critique in a real tool call instead of self-opinion.

In [ ]:
@tool
def lookup(query: str) -> str:
    """Look up a fact."""
    facts = {"n-back": "N-back tasks measure working memory by asking whether the current item matches the one shown n positions earlier."}
    match = next((v for k, v in facts.items() if k in query.lower()), None)
    return match or f"[no canned result for '{query}']"


reflexion_agent = make_reflexion_agent(model, [lookup], max_iterations=2)
print(run_one(
    reflexion_agent,
    "What does an n-back task measure? Call lookup with the exact query 'n-back' "
    "and answer using only what it returns.",
))

## 3. LATS

Explores a few candidate answers per iteration and keeps the best-scored one,
rather than committing to a single trajectory.

In [ ]:
lats_agent = make_lats_agent(model, max_iterations=1, branching=2, score_threshold=0.8)
print(run_one(lats_agent, "Give one concrete example of an n-back trial."))

## Putting one through the full pipeline

Any of the three plugs into `ScannerModel.run(custom_agent=...)` exactly like
any other `ScanningAgent` — here, Basic Reflection (self-contained, no tool
needed), exported to CSV.

In [6]:
RUN_DIR = Path.cwd() / "_reflection_agents_tutorial_run"

task = {
    "tasktype": "survey", "taskname": "reflection_demo",
    "instructions": {"definition": ["Answer thoughtfully in one or two sentences."]},
    "contexts": ["n-back"], "contexts_id": ["nback"], "context_present": False,
    "chain_type": "item", "parser": "0",
    "items": {"nback": [
        {"trcode": "nback_1", "stimulus": "In one sentence, what does an n-back task measure?"},
    ]},
}

card_in = psy.ExpCardInit()
card_in.proj_dir, card_in.projectname = RUN_DIR, "reflection_demo"
card_in.task_file = task
card_in.cogtype, card_in.nsim = "no", 1
card_in.chain_type, card_in.memory = "item", "SingleTurn"

scanner = psy.ScannerModel(expcard=psy.ExpCard(card_in))
scanner.run(custom_agent=reflection_agent)

from psychscanner import to_csv
df = to_csv(scanner, path=RUN_DIR / "reflection_demo.csv")
df.select(["trcode", "pred_resp_raw"])

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_reflection_agents_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_reflection_agents_tutorial_run/reflection_demo/reflection_demo/mock-llm_mock-chat-model_SingleTurn


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...


2026-07-06 12:11:26.756 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 12:11:26.783 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


Saved 1 rows → /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_reflection_agents_tutorial_run/reflection_demo.csv


trcode,pred_resp_raw
str,str
"""rm_1""","""Thank you for the suggestions!…"


## Recap

- Basic Reflection: generate/reflect loop, purely self-critique.
- Reflexion: draft -> execute_tools -> revise, grounded in real tool results.
- LATS: MCTS over candidate trajectories (select/expand/reflect/backpropagate),
  not just one retry chain.
- All three are `ScanningAgent`s like any other — `TaskRunner(scanning_agent=...)`
  directly, or `ScannerModel.run(custom_agent=...)` for a full run.

---
## Further reading

Advanced applications of self-critique/reflection loops:

1. **["Reflexion: Language Agents with Verbal Reinforcement Learning"](https://arxiv.org/abs/2303.11366)** (Shinn et al., 2023) — the paper `make_reflexion_agent` is named after, with the episodic-memory mechanism that lets reflections persist across tasks rather than one draft/revise pass.
2. **["Language Agent Tree Search Unifies Reasoning, Acting, and Planning in Language Models"](https://arxiv.org/abs/2310.04406)** (Zhou et al., 2023) — the LATS paper `make_lats_agent` implements, with the full MCTS select/expand/reflect/backpropagate formulation behind §3 above.